# Method C: Slow & Steady Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/investigate-oasis-sheets-aYOlG/notebooks/method-c-slow-validation.ipynb)

## Theory

The previous bulk download (0.3s delay) returned correct historical data only
in short clusters. Hypothesis: **going slower prevents the collapse** to
current content.

## Approach

1. Start with a **small sample** — mix of known-correct and known-broken revisions
2. Use **longer delays** between API calls (configurable, default 5s)
3. **Canary detection**: if we get the same hash N times in a row, stop
4. **Compare** against previous run's data on Drive
5. **Replace** files on Drive when we get genuinely different (better) content
6. Write **structured progress** to Drive after every single request

## Known State from Previous Run

| Sheet | "Current" Hash | Current Size | Historical Sizes |
|-------|---------------|-------------|------------------|
| Library | `716eacf3...` / `7bd3c0e4...` | 640,702 | 596K-607K |
| Documents | `d1301ded...` / `5fd91297...` | 931,907 / 931,904 | 790K-910K |

In [ ]:
# === Step 0: Auth + Mount ===
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

import google.auth
from google.auth.transport.requests import Request as AuthRequest
creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive: {DRIVE_DIR}')

## Step 1: Test Plan Configuration

**Edit this cell** to adjust what gets tested. Each entry is a revision
number with its expected behavior from the previous run.

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  CONFIGURATION — adjust these for each run                     │
# └─────────────────────────────────────────────────────────────────┘

# Delay between API calls (seconds). Start high, reduce if stable.
REQUEST_DELAY = 5.0

# Stop after this many consecutive identical content hashes.
# A value of 10 means: if 10 revisions in a row return the same
# content.xml hash, assume we've collapsed and stop.
CANARY_THRESHOLD = 10

# Which sheet to test (switch between runs)
TEST_SHEET = 'ubl25_library'
# TEST_SHEET = 'ubl25_documents'

# ── Test plan ─────────────────────────────────────────────────────
# Format: list of (rev_number, group_label)
#   group_label: 'control'  = previously returned correct data
#                'broken'   = previously returned current content
#                'boundary' = at the edge of a cluster
#
# The test runs revisions IN ORDER. We interleave control and broken
# revisions to detect if the collapse happens mid-sequence.

if TEST_SHEET == 'ubl25_library':
    TEST_PLAN = [
        # --- Phase A: re-verify correct cluster 1 (rev 1-39) ---
        (1,   'control'),   # first ever revision
        (10,  'control'),
        (20,  'control'),
        (30,  'control'),
        (39,  'boundary'),  # last correct before collapse

        # --- Phase B: test broken range (rev 40-350) ---
        (40,  'broken'),    # first broken in previous run
        (41,  'broken'),
        (42,  'broken'),
        (43,  'broken'),
        (44,  'broken'),
        (45,  'broken'),
        (50,  'broken'),
        (60,  'broken'),
        (80,  'broken'),
        (100, 'broken'),
        (150, 'broken'),
        (200, 'broken'),
        (250, 'broken'),
        (300, 'broken'),
        (350, 'broken'),    # last before gap

        # --- Phase C: re-verify correct cluster 2 (rev 369-401) ---
        (369, 'control'),   # first correct in cluster 2
        (380, 'control'),
        (395, 'control'),
        (401, 'boundary'),  # last correct before collapse

        # --- Phase D: test broken range 2 (rev 402+) ---
        (402, 'broken'),
        (403, 'broken'),
        (410, 'broken'),
        (450, 'broken'),
        (500, 'broken'),
        (600, 'broken'),
        (800, 'broken'),
        (1000, 'broken'),

        # --- Phase E: high-value revisions (CI workflow range) ---
        (1843, 'broken'),   # Method B: V1+V2
        (1868, 'broken'),   # Method B: V3+V4
        (1999, 'broken'),   # Method B: V5+V6
        (2005, 'broken'),   # Method B: V7-V10 / max rev
    ]
else:  # ubl25_documents
    TEST_PLAN = [
        # --- Phase A: correct cluster 1 (rev 9-47) ---
        (9,   'control'),
        (20,  'control'),
        (35,  'control'),
        (47,  'boundary'),

        # --- Phase B: broken range 1 (rev 48-133) ---
        (48,  'broken'),
        (49,  'broken'),
        (50,  'broken'),
        (60,  'broken'),
        (75,  'broken'),
        (100, 'broken'),
        (133, 'broken'),

        # --- Phase C: correct cluster 2 (rev 146-194) ---
        (146, 'control'),
        (170, 'control'),
        (194, 'boundary'),

        # --- Phase D: broken range 2 (rev 195-1695) ---
        (195, 'broken'),
        (196, 'broken'),
        (200, 'broken'),
        (300, 'broken'),
        (500, 'broken'),
        (1000, 'broken'),
        (1500, 'broken'),
        (1695, 'broken'),

        # --- Phase E: correct cluster 3 (rev 1719-1751) ---
        (1719, 'control'),
        (1740, 'control'),
        (1751, 'boundary'),

        # --- Phase F: broken range 3 + CI revisions ---
        (1752, 'broken'),
        (1793, 'broken'),   # Method B: V1+V2
        (1803, 'broken'),   # Method B: V3
        (1983, 'broken'),   # Method B: V4
        (2000, 'broken'),
        (2190, 'broken'),   # Method B: V5-V7
        (2200, 'broken'),   # Method B: V8
        (2204, 'broken'),   # Method B: V9+V10
    ]

print(f'Test plan: {TEST_SHEET}')
print(f'  {len(TEST_PLAN)} revisions to test')
print(f'  {sum(1 for _, g in TEST_PLAN if g=="control")} control, '
      f'{sum(1 for _, g in TEST_PLAN if g=="broken")} broken, '
      f'{sum(1 for _, g in TEST_PLAN if g=="boundary")} boundary')
print(f'  Delay: {REQUEST_DELAY}s between requests')
print(f'  Canary: stop after {CANARY_THRESHOLD} identical hashes in a row')
est_minutes = len(TEST_PLAN) * REQUEST_DELAY / 60
print(f'  Estimated runtime: ~{est_minutes:.0f} minutes')

In [ ]:
# === Step 2: Helpers ===
import json, time, hashlib, gzip, zipfile, io, re
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError

SHEETS = {
    'ubl25_library':   '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    'ubl25_documents': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
}

# Known "current content" hashes from the previous run.
# If we see these, the API returned the latest version (not historical).
CURRENT_HASHES = {
    'ubl25_library': {
        '716eacf3ebe60d7622e4c9c439e3807b7cf035cd8581ad62c495b8d154064010',
        '7bd3c0e40778115f96c0e5515c093e23e15d1c7d3bbe8b34f29c80f5b97de823',
    },
    'ubl25_documents': {
        'd1301dedea016c962d0dbdb8f287bdd0f60c6911086e5f29f0bca389afa08ac3',
        '5fd9129713bef9336a68ea19821ab406e0e1d6317b1d4a2d7a5ca2aba8a7f8e7',
    },
}

CURRENT_SIZES = {
    'ubl25_library': 640702,
    'ubl25_documents': 931907,
}


def now_iso():
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')


def export_revision_ods(sheet_id, rev_num):
    """Method C: download a specific revision as ODS.
    Returns (ods_bytes, http_status, error_msg)."""
    url = (f'https://docs.google.com/spreadsheets/export'
           f'?id={sheet_id}&revision={rev_num}&exportFormat=ods')
    headers = {'Authorization': f'Bearer {TOKEN}'}
    for attempt in range(3):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=120) as resp:
                data = resp.read()
                if len(data) > 500:
                    return data, 200, None
                return None, 200, f'too small ({len(data)} bytes)'
        except HTTPError as e:
            if e.code in (429, 500, 502, 503) and attempt < 2:
                wait = 2 ** (attempt + 2)  # 4, 8 seconds
                print(f'    retry({e.code}, {wait}s)...', end='', flush=True)
                time.sleep(wait)
                continue
            return None, e.code, str(e.code)
        except Exception as exc:
            if attempt < 2:
                time.sleep(4)
                continue
            return None, 0, str(exc)
    return None, 0, 'max retries'


def ods_content_hash(ods_bytes):
    """Extract content.xml from ODS and SHA256 it."""
    try:
        with zipfile.ZipFile(io.BytesIO(ods_bytes)) as zf:
            return hashlib.sha256(zf.read('content.xml')).hexdigest()
    except Exception:
        return None


print('Helpers ready')

In [ ]:
# === Step 3: Load manifest + spot-check against actual Drive files ===
#
# The manifest has content_hash per revision, but let's VERIFY those
# are correct by reading the actual .ods.gz files from Drive and
# computing the hash ourselves.

manifest_path = DRIVE_DIR / f'manifest-{TEST_SHEET}.json'
prev_data = {}  # rev_num -> {content_hash, ods_size, ...}

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    for r in manifest.get('revisions', []):
        prev_data[r['rev']] = r
    print(f'Loaded previous manifest: {len(prev_data)} revisions')
else:
    print(f'No previous manifest found at {manifest_path}')

current_hashes = CURRENT_HASHES[TEST_SHEET]
ods_dir = DRIVE_DIR / TEST_SHEET


def read_drive_ods_hash(rev_num):
    """Read an existing .ods.gz from Drive, decompress, extract
    content.xml, return its SHA256. Returns None if file missing."""
    gz_path = ods_dir / f'rev-{rev_num}.ods.gz'
    if not gz_path.exists():
        return None
    try:
        ods_bytes = gzip.decompress(gz_path.read_bytes())
        return ods_content_hash(ods_bytes)
    except Exception as e:
        print(f'  [warn] rev-{rev_num} decompress error: {e}')
        return None


# Spot-check: pick a few revisions from the test plan that have
# manifest data, and verify the manifest hash matches the actual file
print(f'\nSpot-checking manifest hashes against actual Drive files...')
spot_checks = [(rev, grp) for rev, grp in TEST_PLAN if rev in prev_data][:8]
spot_ok = 0
spot_mismatch = 0

for rev_num, group in spot_checks:
    manifest_hash = prev_data[rev_num].get('content_hash')
    drive_hash = read_drive_ods_hash(rev_num)

    if drive_hash is None:
        print(f'  rev-{rev_num:>5}: no .ods.gz on Drive (skipped)')
        continue

    if manifest_hash == drive_hash:
        spot_ok += 1
        tag = 'CURRENT' if manifest_hash in current_hashes else 'histor.'
        print(f'  rev-{rev_num:>5}: MATCH  manifest={manifest_hash[:16]}... '
              f'drive={drive_hash[:16]}... [{tag}]')
    else:
        spot_mismatch += 1
        print(f'  rev-{rev_num:>5}: MISMATCH!')
        print(f'           manifest: {manifest_hash[:32]}...')
        print(f'           drive:    {drive_hash[:32]}...')

print(f'\nSpot-check: {spot_ok} OK, {spot_mismatch} mismatch')
if spot_mismatch > 0:
    print('  WARNING: manifest data does not match Drive files!')
    print('  Using Drive files as ground truth for comparison.')
USE_DRIVE_FILES = True  # always compare against actual files when available

## Step 4: Run the Test

Downloads each revision in the test plan with the configured delay.
Writes progress to Drive after **every single request**.

**Canary**: stops if `CANARY_THRESHOLD` consecutive revisions return
the same content hash (= collapsed to current content).

In [ ]:
sheet_id = SHEETS[TEST_SHEET]
ods_dir.mkdir(exist_ok=True)

# Results file — written after every request
results_path = DRIVE_DIR / f'slow-validation-{TEST_SHEET}.json'

# Resume support: load existing results
if results_path.exists():
    results = json.loads(results_path.read_text())
    done_revs = {r['rev'] for r in results.get('tests', [])}
    print(f'Resuming: {len(done_revs)} revisions already tested')
else:
    results = {
        'sheet_key': TEST_SHEET,
        'sheet_id': sheet_id,
        'config': {
            'request_delay': REQUEST_DELAY,
            'canary_threshold': CANARY_THRESHOLD,
        },
        'started': now_iso(),
        'tests': [],
        'summary': {},
    }
    done_revs = set()


def save_results():
    """Write results to Drive."""
    results['last_updated'] = now_iso()
    tests = results['tests']
    results['summary'] = {
        'total_tested': len(tests),
        'total_ok': sum(1 for t in tests if t['status'] == 'ok'),
        'total_error': sum(1 for t in tests if t['status'] == 'error'),
        'historical': sum(1 for t in tests if t.get('is_historical')),
        'current': sum(1 for t in tests if t.get('is_current')),
        'changed_from_prev': sum(1 for t in tests if t.get('changed_from_prev')),
        'replaced_on_drive': sum(1 for t in tests if t.get('replaced_on_drive')),
        'canary_triggered': results.get('canary_triggered', False),
    }
    results_path.write_text(json.dumps(results, indent=2))


# ── Main test loop ──
print(f'\n{"="*70}')
print(f'Starting: {TEST_SHEET} ({len(TEST_PLAN)} revisions, '
      f'{REQUEST_DELAY}s delay)')
print(f'Results: {results_path}')
print(f'{"="*70}\n')

consecutive_same = 0
last_hash = None
canary_triggered = False

for i, (rev_num, group) in enumerate(TEST_PLAN):
    if rev_num in done_revs:
        print(f'  [{i+1}/{len(TEST_PLAN)}] rev-{rev_num:>5} ({group}): skip (already done)')
        continue

    # ── 1. Download from API ──
    t0 = time.time()
    print(f'  [{i+1}/{len(TEST_PLAN)}] rev-{rev_num:>5} ({group:>8}): ',
          end='', flush=True)

    ods_data, http_status, error = export_revision_ods(sheet_id, rev_num)
    dl_elapsed = time.time() - t0

    if not ods_data:
        print(f'ERROR (HTTP {http_status}: {error}) [{dl_elapsed:.1f}s]')
        results['tests'].append({
            'rev': rev_num,
            'group': group,
            'status': 'error',
            'http_status': http_status,
            'error': error,
            'timestamp': now_iso(),
            'elapsed': round(dl_elapsed, 2),
        })
        consecutive_same = 0
        last_hash = None
        save_results()
        time.sleep(REQUEST_DELAY)
        continue

    # ── 2. Hash the new download ──
    content_hash = ods_content_hash(ods_data)
    ods_size = len(ods_data)
    is_current = content_hash in current_hashes
    is_historical = not is_current and content_hash is not None

    # ── 3. Read existing file from Drive for comparison ──
    # We have time (REQUEST_DELAY seconds) so use it wisely
    drive_hash = read_drive_ods_hash(rev_num)
    manifest_hash = prev_data.get(rev_num, {}).get('content_hash')
    # Use Drive file hash as ground truth; fall back to manifest
    prev_hash = drive_hash or manifest_hash
    prev_source = 'drive' if drive_hash else ('manifest' if manifest_hash else None)

    changed_from_prev = (prev_hash is not None and content_hash != prev_hash)
    prev_was_current = prev_hash in current_hashes if prev_hash else None

    # ── 4. Canary check ──
    if content_hash == last_hash:
        consecutive_same += 1
    else:
        consecutive_same = 1
        last_hash = content_hash

    # ── 5. Build result entry ──
    entry = {
        'rev': rev_num,
        'group': group,
        'status': 'ok',
        'content_hash': content_hash,
        'ods_size': ods_size,
        'is_current': is_current,
        'is_historical': is_historical,
        'prev_hash': prev_hash[:24] + '...' if prev_hash else None,
        'prev_source': prev_source,
        'prev_was_current': prev_was_current,
        'changed_from_prev': changed_from_prev,
        'consecutive_same': consecutive_same,
        'timestamp': now_iso(),
        'elapsed': round(dl_elapsed, 2),
        'replaced_on_drive': False,
    }

    # ── 6. Print status ──
    parts = [f'{ods_size:,}b']
    if is_current:
        parts.append('CURRENT')
    else:
        parts.append(f'HISTORICAL ({content_hash[:12]}...)')

    if changed_from_prev:
        if prev_was_current and is_historical:
            parts.append('RECOVERED!')
        elif not prev_was_current and is_current:
            parts.append('REGRESSED')
        else:
            parts.append('CHANGED')
    elif prev_hash:
        parts.append(f'same as prev ({prev_source})')
    parts.append(f'[{dl_elapsed:.1f}s]')
    if consecutive_same > 1:
        parts.append(f'(same x{consecutive_same})')
    print(' '.join(parts))

    # ── 7. Replace on Drive if recovered ──
    if changed_from_prev and is_historical and prev_was_current:
        gz_path = ods_dir / f'rev-{rev_num}.ods.gz'
        gz_data = gzip.compress(ods_data, compresslevel=6)
        gz_path.write_bytes(gz_data)
        entry['replaced_on_drive'] = True
        print(f'         >>> REPLACED on Drive: {gz_path.name} '
              f'({len(gz_data):,} gz bytes)')

    results['tests'].append(entry)
    done_revs.add(rev_num)
    save_results()

    # ── 8. Canary: stop if collapsed ──
    if consecutive_same >= CANARY_THRESHOLD:
        print(f'\n  *** CANARY: {consecutive_same} identical hashes in a row ***')
        print(f'  *** Hash: {content_hash[:32]}...')
        print(f'  *** Method C has collapsed to current content. Stopping.')
        results['canary_triggered'] = True
        results['canary_at_rev'] = rev_num
        results['canary_after_n'] = i + 1
        save_results()
        canary_triggered = True
        break

    # ── 9. Wait (during which Drive I/O in step 3 already happened) ──
    remaining_delay = max(0, REQUEST_DELAY - (time.time() - t0))
    if remaining_delay > 0:
        time.sleep(remaining_delay)

# Final save
results['completed'] = now_iso()
save_results()

if not canary_triggered:
    print(f'\n  All {len(TEST_PLAN)} revisions tested without canary trigger!')

## Step 5: Analyze Results

In [ ]:
# Load and analyze results
results = json.loads(results_path.read_text())
tests = results['tests']

print(f'{"="*70}')
print(f'RESULTS: {TEST_SHEET}')
print(f'{"="*70}')
print(f'  Started:  {results.get("started", "?")}')
print(f'  Completed: {results.get("completed", "?")}')
print(f'  Delay:     {results["config"]["request_delay"]}s')
print()

s = results.get('summary', {})
print(f'  Total tested:       {s.get("total_tested", 0)}')
print(f'  Successful (OK):    {s.get("total_ok", 0)}')
print(f'  Errors:             {s.get("total_error", 0)}')
print(f'  Historical:         {s.get("historical", 0)}')
print(f'  Current (broken):   {s.get("current", 0)}')
print(f'  Changed from prev:  {s.get("changed_from_prev", 0)}')
print(f'  Replaced on Drive:  {s.get("replaced_on_drive", 0)}')
print(f'  Canary triggered:   {s.get("canary_triggered", False)}')

# Show per-group breakdown
print(f'\n  Per-group results:')
for group in ['control', 'broken', 'boundary']:
    group_tests = [t for t in tests if t.get('group') == group and t['status'] == 'ok']
    if not group_tests:
        continue
    hist = sum(1 for t in group_tests if t.get('is_historical'))
    curr = sum(1 for t in group_tests if t.get('is_current'))
    changed = sum(1 for t in group_tests if t.get('changed_from_prev'))
    print(f'    {group:>8}: {len(group_tests)} tested, '
          f'{hist} historical, {curr} current, {changed} changed')

# Detailed results table
print(f'\n{"Rev":>6}  {"Group":>8}  {"Status":>8}  {"Size":>8}  '
      f'{"Content":>8}  {"vs Prev":>10}  {"Same#":>5}  {"Hash (first 16)"}')
print('-' * 95)
for t in tests:
    if t['status'] == 'error':
        print(f'{t["rev"]:>6}  {t["group"]:>8}  {"ERROR":>8}  '
              f'HTTP {t.get("http_status", "?")}')
        continue
    content = 'CURRENT' if t.get('is_current') else 'histor.'
    vs_prev = ''
    if t.get('changed_from_prev'):
        if t.get('prev_was_current') and t.get('is_historical'):
            vs_prev = 'RECOVERED'
        elif not t.get('prev_was_current') and t.get('is_current'):
            vs_prev = 'regressed'
        else:
            vs_prev = 'changed'
    elif t.get('prev_hash'):
        vs_prev = 'same'
    else:
        vs_prev = 'new'
    h = t.get('content_hash', '?')[:16]
    print(f'{t["rev"]:>6}  {t["group"]:>8}  {"ok":>8}  '
          f'{t.get("ods_size",0):>8,}  {content:>8}  '
          f'{vs_prev:>10}  {t.get("consecutive_same",0):>5}  {h}...')

In [ ]:
# === Key Question: Did pacing help? ===

broken_tests = [t for t in tests
                if t.get('group') == 'broken' and t['status'] == 'ok']
broken_historical = [t for t in broken_tests if t.get('is_historical')]
broken_current = [t for t in broken_tests if t.get('is_current')]

print(f'\n{"="*70}')
print(f'KEY FINDING: Did slower pacing recover historical data?')
print(f'{"="*70}')
print(f'  Previously-broken revisions tested:  {len(broken_tests)}')
print(f'  Now returning HISTORICAL content:    {len(broken_historical)}')
print(f'  Still returning CURRENT content:     {len(broken_current)}')
print()

if len(broken_historical) > 0:
    pct = 100 * len(broken_historical) / len(broken_tests)
    print(f'  >>> YES! {pct:.0f}% of previously-broken revisions now return '
          f'historical data!')
    print(f'  >>> Pacing hypothesis CONFIRMED.')
    print(f'\n  Recovered revisions:')
    for t in broken_historical:
        print(f'    rev-{t["rev"]:>5}: {t["ods_size"]:,} bytes, '
              f'hash={t["content_hash"][:20]}...')
elif len(broken_tests) == 0:
    print('  No broken revisions were tested (all errored out).')
else:
    print(f'  >>> NO — all previously-broken revisions still return current content.')
    print(f'  >>> Pacing alone does not fix Method C.')
    print(f'  >>> The collapse is not a rate-limiting issue.')

# Control group verification
control_tests = [t for t in tests
                 if t.get('group') in ('control', 'boundary') and t['status'] == 'ok']
control_same = sum(1 for t in control_tests if not t.get('changed_from_prev'))
control_changed = sum(1 for t in control_tests if t.get('changed_from_prev'))

print(f'\n  Control group ({len(control_tests)} revisions):')
print(f'    Same as previous run: {control_same}')
print(f'    Changed from previous: {control_changed}')
if control_same == len(control_tests):
    print(f'    >>> Control group is consistent — our hashes are reliable.')
elif control_changed > 0:
    print(f'    >>> WARNING: {control_changed} control revisions returned '
          f'different data! Method C is non-deterministic.')

# nothing further - check slow-validation-{sheet}.json on Drive for results